## Dataset:

For more advanced instructions check cust_xview.yaml

https://challenge.xviewdataset.org


Source:\
@misc{lam2018xview,
      title={xView: Objects in Context in Overhead Imagery},
      author={Darius Lam and Richard Kuzma and Kevin McGee and Samuel Dooley and Michael Laielli and Matthew Klaric and Yaroslav Bulatov and Brendan McCord},
      year={2018},
      eprint={1802.07856},
      archivePrefix={arXiv},
      primaryClass={cs.CV}
}

In [ ]:
import matplotlib.pyplot as plt

#### For Windows:

In [ ]:
# Source - https://stackoverflow.com/a/58295505
# Posted by wimworks, modified by community. See post 'Timeline' for change history
# Retrieved 2026-05-04, License - CC BY-SA 4.0

# import ctypes

# libbytiff = ctypes.CDLL("libtiff-5.dll")
# libbytiff.TIFFSetWarningHandler.argtypes = [ctypes.c_void_p]
# libbytiff.TIFFSetWarningHandler.restype = ctypes.c_void_p
# libbytiff.TIFFSetWarningHandler(None)

### For Linux:

In [ ]:
import ctypes

# Using find_library is often safer than hardcoding versions
from ctypes.util import find_library
lib_path = find_library("tiff")
if not lib_path:
    # Fallback to common Linux name if find_library fails
    lib_path = "libtiff.so.5"
libbytiff = ctypes.CDLL(lib_path)
libbytiff.TIFFSetWarningHandler.argtypes = [ctypes.c_void_p]
libbytiff.TIFFSetWarningHandler.restype = ctypes.c_void_p
libbytiff.TIFFSetWarningHandler(None)

In [ ]:
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

# model = YOLO('yolo26l.pt') 

model = YOLO('yolo11n.pt')  # nano yolo11 --> simplest model
# download the xview dataset
model.train(
            # configs
            data='cust_xview.yaml', 
            project = 'runs/detect', # only useful for first path initialization
            device = "cuda",
            classes = [52], # the construction site class
            single_cls=True,
            pretrained=False,
            amp=True,  

            # depending on resources
            # fraction = .5, # a % of the dataset
            batch = 16, # can  also be set to a 0.x percent of gpu usage or to -1 for automatic batching
            epochs = 2, 
            imgsz = 384,

            # transformations
            degrees=180.0,          
            flipud=0.5,            
            fliplr=0.5,


            optimizer = "auto",
            mosaic = 0.5, #  mosaic augmentation reduciton
            conf = 0.15,
            # https://docs.ultralytics.com/usage/cfg/#train-settings
            )

In [ ]:
%matplotlib inline

# model = YOLO('yolo26l.pt') 


from pathlib import Path
import cv2
val_path = Path('datasets/xView/images/train')
image_ids = [372, 1224, 1058, 1460]

fig, axes = plt.subplots(1, 4, figsize=(12, 6))
for ax, img_id in zip(axes, image_ids):
    img_path = val_path / str(img_id)
    results = model.predict(source=f"{str(img_path)}.tif", imgsz=1024, conf=0.30, save=False)[0]
    annotated_img = results.plot(labels=True, boxes=True, conf=True)
    annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    ax.set_title(f"Predicitons on img: {img_id}", fontsize=8)
    ax.imshow(annotated_img)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Important to execute at the beginning to automatically split using ultralytics autosplit

In [ ]:
# from ultralytics.data.split import autosplit
# from pathlib import Path
# import shutil

# dir = Path('datasets/xView')

# # Convert the labels
# def convert_labels():
#     # This function's logic is based on the official Ultralytics xView.yaml file
#     import json, numpy as np
#     from PIL import Image
#     from ultralytics.utils.ops import xyxy2xywhn
#     from ultralytics.utils import TQDM

#     fname = dir / "xView_train.geojson"
#     path = fname.parent
#     with open(fname, encoding="utf-8") as f:
#         print(f"Loading {fname}...")
#         data = json.load(f)

#     labels = path / "labels" / "train"
#     shutil.rmtree(labels, ignore_errors=True)
#     labels.mkdir(parents=True, exist_ok=True)

#     xview_class2index = [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 0, 1, 2, -1, 3, -1, 4, 5, 6, 7, 8, -1, 9, 10, 11, 12, 13, 14, 15, -1, -1, 16, 17, 18, 19, 20, 21, 22, -1, 23, 24, 25, -1, 26, 27, -1, 28, -1, 29, 30, 31, 32, 33, 34, 35, 36, 37, -1, 38, 39, 40, 41, 42, 43, 44, 45, -1, -1, -1, -1, 46, 47, 48, 49, -1, 50, 51, -1, 52, -1, -1, -1, 53, 54, -1, 55, -1, -1, 56, -1, 57, -1, 58, 59]

#     shapes = {}
#     for feature in TQDM(data["features"], desc=f"Converting {fname}"):
#         p = feature["properties"]
#         if p["bounds_imcoords"]:
#             image_id = p["image_id"]
#             image_file = path / "train_images" / image_id
#             if image_file.exists():
#                 try:
#                     box = np.array([int(num) for num in p["bounds_imcoords"].split(",")])
#                     assert box.shape[0] == 4, f"incorrect box shape {box.shape[0]}"
#                     cls = p["type_id"]
#                     cls = xview_class2index[int(cls)]
#                     assert 59 >= cls >= 0, f"incorrect class index {cls}"

#                     if image_id not in shapes:
#                         shapes[image_id] = Image.open(image_file).size
#                     box = xyxy2xywhn(box[None].astype(float), w=shapes[image_id][0], h=shapes[image_id][1], clip=True)
#                     with open((labels / image_id).with_suffix(".txt"), "a", encoding="utf-8") as f:
#                         f.write(f"{cls} {' '.join(f'{x:.6f}' for x in box[0])}\n")
#                 except Exception as e:
#                     print(f"WARNING: skipping one label for {image_file}: {e}")

# convert_labels()

# # Move and rename image folders to the final structure
# images = Path(dir / "images")
# images.mkdir(parents=True, exist_ok=True)
# Path(dir / "train_images").rename(dir / "images" / "train")
# Path(dir / "val_images").rename(dir / "images" / "val")

# # Automatically split the training images into train/val sets (e.g., 90%/10%)
# autosplit(dir / "images" / "train")